# Rapprochement DPE aout 2026

In [7]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [8]:
usecols = ['numero_dpe', 'identifiant_ban', 'code_postal_ban', 'code_postal_brut', 'id_rnb', 'provenance_id_rnb']
dtype={'numero_dpe': 'string', 'identifiant_ban': 'string', 'code_postal_ban': 'string', 'code_postal_brut': 'string', 'id_rnb': 'string', 'provenance_id_rnb': 'string'}

# usecols = ['numero_dpe', 'id_rnb', 'provenance_id_rnb']
# dtype={'numero_dpe': 'string', 'id_rnb': 'string', 'provenance_id_rnb': 'string'}

In [3]:
df_tertiaire = pd.read_csv('notebooks/rapprochements/DPE/2026/data/dpe01tertiaire.csv', sep=',', usecols=usecols, dtype=dtype)
df_tertiaire['file'] = 'tertiaire'

In [4]:
df_existant = pd.read_csv('notebooks/rapprochements/DPE/2026/data/dpe03existant.csv', sep=',', usecols=usecols, dtype=dtype)
df_existant['file'] = 'existant'

KeyboardInterrupt: 

In [ ]:
df_neuf = pd.read_csv('notebooks/rapprochements/DPE/2026/data/dpe02neuf.csv', sep=',', usecols=usecols, dtype=dtype)
df_neuf['file'] = 'neuf'

In [ ]:
df = pd.concat([df_tertiaire, df_neuf, df_existant])

In [ ]:
df.shape

(17255757, 7)

In [ ]:
df.head()

,numero_dpe,id_rnb,provenance_id_rnb,code_postal_ban,identifiant_ban,code_postal_brut,file
0,2618T0063102M,<NA>,<NA>,18200,18197_0540,18200,tertiaire
1,2606T0031071B,<NA>,<NA>,06210,06079_0602_00026,06210,tertiaire
2,2665T0010661I,<NA>,<NA>,65000,65440_1990_00040,65000,tertiaire
3,2692T0076695D,<NA>,<NA>,92000,92050_4448,92000,tertiaire
4,2675T0079080F,8PAF1D23GEJC,Logiciel,75019,75119_8241_00058,75020,tertiaire


In [ ]:
# create 100 subfiles in the dpe_existant folder, using the number of rows in the dataframe
for i in range(0, 100):
    df.iloc[i*df.shape[0]//100:(i+1)*df.shape[0]//100].to_csv(f'notebooks/rapprochements/DPE/2026/sub_files/dpe-{i}.csv', index=False)

In [9]:
# le rapprochement

import os
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
from django.db import connection
from concurrent.futures import ThreadPoolExecutor
import numpy as np

def get_rnb_id(row):
    cursor = connection.cursor()

    if row['code_postal_ban'] != row['code_postal_brut']:
        return []
    
    ban_id = row['identifiant_ban']

    sql = f"""
            with rnb_ids as (
            select
                rnb_id
            from
                batid_buildingaddressesreadonly bb
            left join batid_building bb2 on
                bb2.id = bb.building_id
            where
                address_id = '{ban_id}'
                and ST_AREA(shape::geography) > 25)
            select
                array_agg(rnb_id)
            from
                rnb_ids;
    """
    cursor.execute(sql)
    result = cursor.fetchone()
    return result[0] if result[0] is not None else []

def execute(df):
    df_copy = df.copy()
    df_copy['rnb_id_rappro'] = df_copy.apply(get_rnb_id, axis=1)
    return df_copy


def process_sub_file(i):
    print(f"processing file {i}")
    if not os.path.exists(f'notebooks/rapprochements/DPE/2026/sub_files_results/dpe-{i}-result.csv'):
        df_sub_file = pd.read_csv(f'notebooks/rapprochements/DPE/2026/sub_files/dpe-{i}.csv', sep=',')
        max_workers = 50
        dfs = np.array_split(df_sub_file, max_workers)

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            results_sub_file = executor.map(execute, dfs)
            df_result = pd.concat(results_sub_file)
            df_result.to_csv(f'notebooks/rapprochements/DPE/2026/sub_files_results/dpe-{i}-result.csv', index=False)

for i in range(100):
    process_sub_file(i)

processing file 0


AttributeError: 'numpy.ndarray' object has no attribute 'apply'

In [ ]:
# aggregation des sous fichiers de résultats en un seul
dfs_result = []

for i in range(100):
    df_result = pd.read_csv(f'notebooks/rapprochements/DPE/2025/sub_files_results/dpe-{i}-result.csv', usecols=['numero_dpe', 'file', 'rnb_id_rappro', 'id_rnb', 'provenance_id_rnb'])
    dfs_result.append(df_result)

df_final = pd.concat(dfs_result)
df_final['rnb_id_rappro'] = df_final['rnb_id_rappro'].apply(eval)


/tmp/ipykernel_2074/3264209841.py:4: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_result = pd.read_csv(f'notebooks/rapprochements/DPE/2025/sub_files_results/dpe-{i}-result.csv', usecols=['numero_dpe', 'file', 'rnb_id_rappro', 'id_rnb', 'provenance_id_rnb'])
/tmp/ipykernel_2074/3264209841.py:4: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_result = pd.read_csv(f'notebooks/rapprochements/DPE/2025/sub_files_results/dpe-{i}-result.csv', usecols=['numero_dpe', 'file', 'rnb_id_rappro', 'id_rnb', 'provenance_id_rnb'])
/tmp/ipykernel_2074/3264209841.py:4: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_result = pd.read_csv(f'notebooks/rapprochements/DPE/2025/sub_files_results/dpe-{i}-result.csv', usecols=['numero_dpe', 'file', 'rnb_id_rappro', 'id_rnb', 'provenance_id_rnb'])
/tmp/ipykernel_2074/3264209841.py:4: Dty

In [ ]:
# sauvegarde des résultats
df_final.to_csv('notebooks/rapprochements/DPE/2025/results_DPE_RNB.csv', index=False)